# BioScout plots — SO muscle peak forces per task

Quick comparison of Static Optimization muscle-group **peak forces** per task (SJ, SLS back, SLS front) for the participants added in commit `7a2a6749` (038, 039, 040, 041, 044, 047, 054, 073, 075, 076, 077).

Outputs go to `results/peak_forces/` (per-trial CSV, summary CSV, figure PNG).

Note: only 073/075/076/077 have SO results for the SLS trials; 038–054 currently have SO for SJ only.

In [1]:
import glob, os, json
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT = os.path.abspath('..')  # repo root when run from code/
PIDS = ['038','039','040','041','044','047','054','073','075','076','077']

GROUPS = {
 'Vasti': ['vasint','vaslat','vasmed'],
 'Gluteus maximus': ['glmax1','glmax2','glmax3'],
 'Gluteus medius': ['glmed1','glmed2','glmed3'],
 'Gluteus minimus': ['glmin1','glmin2','glmin3'],
 'Hamstrings': ['bflh','bfsh','semimem','semiten'],
 'Adductors': ['addbrev','addlong','addmagDist','addmagIsch','addmagMid','addmagProx','grac'],
 'Triceps surae': ['gaslat','gasmed','soleus'],
 'Iliopsoas': ['iliacus','psoas'],
 'Rectus femoris': ['recfem'],
 'Tibialis anterior': ['tibant'],
 'Piriformis': ['piri'],
}

def read_sto(p):
    with open(p) as f:
        lines = f.readlines()
    h = next(i for i,l in enumerate(lines) if l.strip()=='endheader')
    return pd.read_csv(p, sep='\t', skiprows=h+1)

def task_of(trial):
    if trial.startswith('SJ'): return 'SJ'
    if trial.startswith('SLS_back'): return 'SLS back'
    if trial.startswith('SLS_front'): return 'SLS front'
    if trial.lower().startswith('squat'): return 'Squat'
    return None

In [2]:
rows = []
for pid in PIDS:
    for p in glob.glob(f'{ROOT}/c3dfiles/{pid}/*/SO_StaticOptimization_force.sto'):
        trial = os.path.basename(os.path.dirname(p))
        task = task_of(trial)
        if task is None: continue
        df = read_sto(p)
        for g, ms in GROUPS.items():
            best = 0.0
            for side in ('_r','_l'):
                cols = [m+side for m in ms if m+side in df.columns]
                if cols:
                    best = max(best, df[cols].sum(axis=1).max())
            rows.append(dict(pid=pid, trial=trial, task=task, group=g, peak_N=best))

d = pd.DataFrame(rows)
os.makedirs(f'{ROOT}/results/peak_forces', exist_ok=True)
d.to_csv(f'{ROOT}/results/peak_forces/muscle_peak_forces_per_trial.csv', index=False)

# mean of per-trial peaks per participant, then mean across participants
per_pid = d.groupby(['task','group','pid'])['peak_N'].mean().reset_index()
summ = per_pid.groupby(['task','group'])['peak_N'].agg(['mean','std','count']).reset_index()
summ.to_csv(f'{ROOT}/results/peak_forces/muscle_peak_forces_summary.csv', index=False)

In [3]:
tasks = sorted(d['task'].unique())
order = summ.groupby('group')['mean'].mean().sort_values(ascending=False).index.tolist()
fig, axes = plt.subplots(1, len(tasks), figsize=(5*len(tasks), 5), sharey=True)
if len(tasks)==1: axes=[axes]
for ax, t in zip(axes, tasks):
    s = summ[summ['task']==t].set_index('group').reindex(order)
    ax.barh(range(len(order)), s['mean'], xerr=s['std'], color='#2b6cb0')
    ax.set_yticks(range(len(order))); ax.set_yticklabels(order)
    ax.invert_yaxis(); ax.set_title(t); ax.set_xlabel('Peak force (N)')
n = d.groupby('task')['pid'].nunique().to_dict()
fig.suptitle('SO muscle-group peak forces per task — participants ' + ', '.join(PIDS), fontsize=10)
fig.tight_layout()
fig.savefig(f'{ROOT}/results/peak_forces/muscle_peak_forces_per_task.png', dpi=200)
print(summ.to_string(index=False))
print('participants per task:', n)

     task             group         mean         std  count
       SJ         Adductors  3459.792372  402.662718     11
       SJ   Gluteus maximus  1982.907271  719.787184     11
       SJ    Gluteus medius  2533.881700  322.559044     11
       SJ   Gluteus minimus  1301.133839   81.437068     11
       SJ        Hamstrings  4545.497384  407.027899     11
       SJ         Iliopsoas  2423.871743  312.991256     11
       SJ        Piriformis   818.121324  211.808418     11
       SJ    Rectus femoris  2550.901149  117.928467     11
       SJ Tibialis anterior  1527.758251  299.720963     11
       SJ     Triceps surae  7462.494406 2105.454596     11
       SJ             Vasti  7134.962045 2195.517032     11
 SLS back         Adductors  3424.521688  349.742045      4
 SLS back   Gluteus maximus  3075.874325  172.782168      4
 SLS back    Gluteus medius  2879.227532  149.750620      4
 SLS back   Gluteus minimus  1295.593991   79.998944      4
 SLS back        Hamstrings  4570.068976

## Split by participant group (CON / FAIM / FAIS)

Group labels from `ParticipantData and Labelling.xlsx` (Demographics sheet). Only the FAIS participants (073/075/076/077) have SO for the SLS trials, so CON/FAIM appear only in SJ.

In [4]:
import numpy as np

ROOT = os.path.abspath('..')
OUT = f'{ROOT}/results/peak_forces'
d = pd.read_csv(f'{OUT}/muscle_peak_forces_per_trial.csv', dtype={'pid':str})

gr = pd.read_excel(f'{ROOT}/ParticipantData and Labelling.xlsx', sheet_name='Demographics')
gr['pid'] = gr['Subject'].apply(lambda s: f"{int(s):03d}" if pd.notna(s) else None)
d = d.merge(gr[['pid','Group']], on='pid')

per_pid = d.groupby(['task','Group','group','pid'])['peak_N'].mean().reset_index()
summ = per_pid.groupby(['task','Group','group'])['peak_N'].agg(['mean','std','count']).reset_index()
summ.to_csv(f'{OUT}/muscle_peak_forces_summary_by_group.csv', index=False)

tasks = sorted(d['task'].unique())
order = summ.groupby('group')['mean'].mean().sort_values(ascending=False).index.tolist()
groups = ['CON','FAIM','FAIS']
colors = {'CON':'#4a4a4a','FAIM':'#2b6cb0','FAIS':'#d97706'}

fig, axes = plt.subplots(1, len(tasks), figsize=(5.5*len(tasks), 6), sharey=True)
if len(tasks)==1: axes=[axes]
y = np.arange(len(order)); h = 0.25
for ax, t in zip(axes, tasks):
    for i, G in enumerate(groups):
        s = summ[(summ['task']==t)&(summ['Group']==G)].set_index('group').reindex(order)
        n = int(s['count'].max()) if s['count'].notna().any() else 0
        ax.barh(y+(i-1)*h, s['mean'], height=h, xerr=s['std'],
                color=colors[G], label=f'{G} (n={n})' if n else f'{G} (n=0)',
                error_kw=dict(lw=0.8))
    ax.set_yticks(y); ax.set_yticklabels(order); ax.invert_yaxis()
    ax.set_title(t); ax.set_xlabel('Peak force (N)'); ax.legend(fontsize=8)
fig.suptitle('SO muscle-group peak forces per task, split by participant group', fontsize=11)
fig.tight_layout()
fig.savefig(f'{OUT}/muscle_peak_forces_per_task_by_group.png', dpi=200)
print(summ.pivot_table(index=['task','group'], columns='Group', values='mean').round(0).to_string())

Group                           CON    FAIM     FAIS
task      group                                     
SJ        Adductors          3496.0  3683.0   3256.0
          Gluteus maximus    2283.0  2139.0   1566.0
          Gluteus medius     2567.0  2671.0   2398.0
          Gluteus minimus    1264.0  1329.0   1318.0
          Hamstrings         4483.0  4658.0   4524.0
          Iliopsoas          2306.0  2292.0   2640.0
          Piriformis          869.0   897.0    708.0
          Rectus femoris     2567.0  2581.0   2512.0
          Tibialis anterior  1442.0  1472.0   1655.0
          Triceps surae      8416.0  7598.0   6408.0
          Vasti              7942.0  7504.0   6051.0
SLS back  Adductors             NaN     NaN   3425.0
          Gluteus maximus       NaN     NaN   3076.0
          Gluteus medius        NaN     NaN   2879.0
          Gluteus minimus       NaN     NaN   1296.0
          Hamstrings            NaN     NaN   4570.0
          Iliopsoas             NaN     NaN   